# DP_task_RAG · Advanced + Agentic RAG over Harvey LAB (antitrust) on a free Colab T4

**Runtime → Change runtime type → T4 GPU**, then run top to bottom (~15 min plus the eval you choose).

| Section | What happens | GPU / LLM |
|---|---|---|
| 1. Setup | clone, install, load Claude credentials from Colab Secrets | – |
| 2. Indexes | BGE-M3 dense + BM25 for 3 text variants: `raw` / `header` / `ctx` (Contextual Retrieval) | T4, ~1.5 min per variant |
| 3. Observability | Arize Phoenix in the notebook, LangGraph auto-instrumented + manual retriever/LLM spans | – |
| 4. Demo | the agent on a *simple*, a *complex* and a *direct* question, showing its decision log | T4 + Claude |
| 5. Evaluation | retrieval ablations, CRAG calibration, generation eval (naive vs. single-shot vs. agent), Adaptive-RAG ablation | T4 + Claude |

**Credentials** (🔑 Secrets, enable notebook access): `CLAUDE_CODE_OAUTH_TOKEN` (from `claude setup-token`, uses your
Claude subscription through `claude_agent_sdk`) **or** `ANTHROPIC_API_KEY` (fallback, Anthropic API). Nothing is printed.

The one-off Contextual Retrieval job (Qwen2.5-3B on vLLM, ~80–90 min) lives in `01_contextualize_colab.ipynb`. Its
output `artifacts/contexts.jsonl` is committed, so this notebook never re-runs it.

## 1. Setup

In [ ]:
import os, subprocess, sys, json
REPO = "https://github.com/nrusnac/DP_task_RAG.git"
if not os.path.exists("/content/DP_task_RAG"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/DP_task_RAG"], check=True)
os.chdir("/content/DP_task_RAG")
subprocess.run(["git", "pull", "-q"], check=True)
!pip -q install -e ".[obs]"
# an editable install is only picked up by NEW processes; make it importable in this running kernel
sys.path.insert(0, "/content/DP_task_RAG/src")


In [ ]:
import hrag.config  # sets JAX_PLATFORMS=cpu before bm25s is imported (Colab's GPU JAX would grab 75% of VRAM)
from hrag import llm
print("secrets found:", llm.load_colab_secrets(), "| backend:", llm.backend())
print(open("artifacts/ingest_stats.json").read())

## 2. Indexes
Text variants for the Contextual Retrieval ablation: `raw` (chunk only), `header` (deterministic
`[doc title] (type) › section` + chunk), and `ctx` (header + Qwen-written context + chunk). If `contexts.jsonl` isn't
in the repo yet, it's looked up on Drive (where notebook 01 writes it).

In [ ]:
from pathlib import Path
ctx = Path("artifacts/contexts.jsonl")
if not ctx.exists() and Path("/content/drive/MyDrive/dp_task_rag/contexts.jsonl").exists():
    import shutil; shutil.copy("/content/drive/MyDrive/dp_task_rag/contexts.jsonl", ctx)
variants = ["raw", "header"] + (["ctx"] if ctx.exists() else [])
print("building:", variants)
!python -m hrag.index.build --variants {" ".join(variants)}

## 3. Observability (Phoenix)

In [ ]:
from hrag.obs import tracing
print("Phoenix UI →", tracing.setup(project="dp-task-rag"))

## 4. Demo: the agentic module
Watch the `log`: the Adaptive-RAG route, the CRAG verdict per sub-question (with the best rerank score), corrective
rewrites, and sufficiency iterations. Every step is also a span in Phoenix.

In [ ]:
from hrag.agent.graph import answer
from IPython.display import Markdown, display
VARIANT = "ctx" if "ctx" in variants else "header"

async def show(q, task_id, **kw):
    r = await answer(q, task_id=task_id, variant=VARIANT, **kw)
    display(Markdown(f"**Q:** {q}\n\n**route:** `{r['route']}` · **LLM calls:** {r['llm_calls']} · **{r['seconds']} s**\n\n"
                     + "\n".join(f"- {l}" for l in r["log"]) + "\n\n---\n" + (r["answer"] or "")
                     + "\n\n**Sources:** " + ", ".join(f"[{c['n']}] {c['doc']}" for c in r["citations"])))
    return r

T = "analyze-iss-antitrust-transaction-structure"
r1 = await show("What are the post-merger HHI and the HHI delta for flame retardants in the South Central region?", T)

In [ ]:
r2 = await show("Which product markets raise presumptive concerns under the 2023 Merger Guidelines, and what remedies "
                "and HSR filing strategy should the client consider?", T)

In [ ]:
r3 = await show("What can you help me with?", T)

**CRAG's corrective path:** a question the data room can't answer should be graded *incorrect*, rewritten (keywords + HyDE), retried, and then **abstained on** instead of answered from look-alike evidence.

In [ ]:
r4 = await show("What annual membership fee does the target pay for its CEO's country club, and who approved it?", T)

## 5. Evaluation
The eval set (`artifacts/eval/qa.jsonl`) was built from Harvey rubric criteria: Claude rewrote each criterion as a
question plus verbatim key facts, and chunks containing those facts are the **silver gold labels**
(`python -m hrag.eval.build_qa`). See `docs/05-evaluation.md`.

In [ ]:
from hrag.eval import run_eval
qa = run_eval.load_qa()
print(len(qa), "QA items |", sum(bool(i["gold"]) for i in qa), "with silver retrieval labels")
ret = run_eval.retrieval(variants)          # 5a. retrieval ablations: no LLM calls

In [ ]:
import pandas as pd
pd.DataFrame(ret).T[["recall@1", "recall@5", "recall@10", "mrr", "ndcg@10", "foreign@5", "ms_per_query"]]

In [ ]:
cal = run_eval.calibrate(VARIANT)           # 5b. CRAG thresholds: is the top rerank score predictive of success?

In [ ]:
# 5c. generation eval: naive RAG vs. advanced single-shot RAG vs. the agent (Harvey judge + faithfulness)
N = 30
gen = await run_eval.generation(n=N, systems=["naive", "single_shot", "agent"], variant=VARIANT)
pd.DataFrame(gen).T

In [ ]:
# 5d. Adaptive-RAG ablation: adaptive routing vs. forcing every query down one path
abl = await run_eval.generation(n=20, systems=["agent:simple", "agent:complex", "agent"], variant=VARIANT,
                                judge_faith=False)
pd.DataFrame(abl).T[["pass_rate", "avg_llm_calls", "avg_seconds", "routes"]]

5e. Unanswerable questions: each question asked in ANOTHER matter's data room. A correct system declines instead of answering from look-alike evidence.

In [ ]:
neg = await run_eval.negatives(n=12, systems=["single_shot", "agent"], variant=VARIANT)
pd.DataFrame(neg).T

In [ ]:
print("LLM usage this session:", llm.STATS.snapshot())
print(json.dumps({"retrieval": ret, "calibration": cal["suggested"], "generation": gen, "adaptive_ablation": abl, "unanswerable": neg},
                 indent=1, default=str)[:6000])

In [ ]:
# Save the results: download the zip and unzip it into the repo's artifacts/eval/ (also copied to Drive if mounted)
import shutil, os
shutil.make_archive("/content/eval_results", "zip", "/content/DP_task_RAG/artifacts", "eval")
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs("/content/drive/MyDrive/dp_task_rag", exist_ok=True)
    shutil.copy("/content/eval_results.zip", "/content/drive/MyDrive/dp_task_rag/eval_results.zip")
from google.colab import files
files.download("/content/eval_results.zip")